## Этап 1 - Подключение библиотек и базовые настройки

В данном разделе подключаются библиотеки для работы с таблицами, геометрией объектов, расчета метрик качества и организации групповой кросс-валидации. В качестве основной модели рассматривается `CatBoostRegressor` — алгоритм градиентного бустинга, который особенно удобен при работе с табличными данными, содержащими как числовые, так и категориальные признаки.

Цель эксперимента — проверить, насколько эффективно CatBoost сможет предсказывать высоту здания по признакам, полученным после пространственного объединения объектов, и сравнить результаты между упрощенным и полным набором признаков.

---

Кроме того фиксируются основные константы эксперимента: путь к входному parquet-файлу, значение `random_state` для воспроизводимости результатов, число фолдов в кросс-валидации и размер пространственной сетки, по которой далее будут формироваться группы объектов.

Также задается список основных столбцов, которые сохраняются из исходного датасета для дальнейшего моделирования. В него входят признаки качества сопоставления, агрегаты по геометрии, статистики по соседним объектам и целевая переменная — наблюдаемая высота здания.

In [ ]:
import numpy as np
import pandas as pd

from shapely import wkb

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold

from catboost import CatBoostRegressor
from pathlib import Path 
from itertools import product

In [ ]:
DATA_PATH = Path("../data/interim/merged_buildings_by_geometry.parquet")
RANDOM_STATE = 42
N_SPLITS = 5
GRID_SIZE = 1000

main_cols = [
    "component_id",
    "match_type",
    "match_confidence",
    "geometry_source",

    "target_height",
    "target_height_is_observed",
    "target_height_source",
    "target_height_source_detail",
    "target_height_reliability",

    "n_a",
    "n_b",
    "uids_a",
    "uids_b",

    "n_edges_ab",
    "max_iou",
    "mean_iou",
    "max_overlap_a",
    "max_overlap_b",
    "min_dist_m",

    "sum_area_a",
    "sum_area_b",
    "union_area_a",
    "union_area_b",
    "union_area_all",

    "n_b_with_height",
    "median_height_b",
    "median_stairs_b",
    "median_avg_floor_height_b",
    "mode_purpose_b",

    "n_neighbors_50m",
    "n_neighbors_obs_height_50m",
    "neighbor_height_mean_50m",
    "neighbor_height_median_50m",
    "neighbor_height_min_50m",
    "neighbor_height_max_50m",
    "neighbor_height_std_50m",
    "neighbor_height_q25_50m",
    "neighbor_height_q75_50m",

    "n_neighbors_100m",
    "n_neighbors_obs_height_100m",
    "neighbor_height_mean_100m",
    "neighbor_height_median_100m",
    "neighbor_height_min_100m",
    "neighbor_height_max_100m",
    "neighbor_height_std_100m",
    "neighbor_height_q25_100m",
    "neighbor_height_q75_100m",

    "rep_geometry",
]

## Этап 2 - Подготовка к обучению 

**Преобразование геометрии из WKB-формата**

Так как геометрия объектов хранится в бинарном виде, перед любыми пространственными вычислениями ее необходимо преобразовать в полноценные геометрические объекты `shapely`.

Эта операция нужна для последующего определения центроидов зданий. Именно через центроиды далее будет строиться пространственная группировка, необходимая для честной валидации модели.

---

**Формирование пространственных групп по координатам центроидов**

После восстановления геометрии для каждого объекта вычисляется его центроид. Координаты центроидов используются для отнесения объекта к укрупненной пространственной ячейке регулярной сетки.

Каждому зданию присваивается идентификатор пространственной группы. Такая группировка нужна для применения `GroupKFold`: в этом случае пространственно близкие объекты попадают в один и тот же блок, что снижает риск утечки информации между обучающей и тестовой выборками.

---

**Обработка пропусков в соседских статистиках**

Соседские признаки, связанные с распределением высот вокруг здания, могут отсутствовать, если в заданном радиусе не найдено достаточного числа релевантных объектов. Чтобы эта ситуация сама по себе не терялась, для каждого такого признака создается отдельный бинарный индикатор наличия значения.

После этого числовые пропуски в соседских статистиках заполняются нулями. В результате модель получает сразу две составляющие информации: само значение признака и сигнал о том, было ли оно реально доступно в исходных данных.

---

**Выбор метрик качества модели**

Для оценки качества предсказаний используются метрики `MSE`, `RMSE`, `MAE` и `R²`. Они позволяют оценить ошибку модели как в квадратичной форме, так и в абсолютных отклонениях, а также понять, какую долю изменчивости целевой переменной объясняет модель.

Использование сразу нескольких метрик важно, поскольку одна и та же модель может выглядеть по-разному в зависимости от способа измерения ошибки. В задачах регрессии особенно полезно смотреть на `RMSE` и `MAE` совместно.

---

**Формирование наборов признаков для сравнения**

На данном этапе задаются два варианта признакового пространства. Первый — `baseline` — включает только базовые характеристики здания: агрегированную этажность, среднюю высоту этажа, общую площадь объединенной геометрии и тип назначения объекта.

Второй — `full` — представляет собой расширенный набор признаков. В него входят признаки качества сопоставления объектов, агрегаты по геометрии, сведения о количестве объединяемых сущностей и подробные статистики по соседям в радиусах 50 и 100 метров, а также индикаторы наличия этих статистик.

Такое сравнение позволяет проверить, насколько полезен для модели расширенный пространственный контекст по сравнению с компактным базовым описанием объекта.




In [4]:
def load_wkb_geometry(value):
    if value is None or pd.isna(value):
        return None

    if isinstance(value, memoryview):
        value = value.tobytes()
    elif isinstance(value, bytearray):
        value = bytes(value)

    return wkb.loads(value)

In [5]:
def build_spatial_groups(df, geom_col="rep_geometry", grid_size=1000):
    geoms = df[geom_col].apply(load_wkb_geometry)
    centroids = geoms.apply(lambda g: g.centroid if g is not None else None)

    x = centroids.apply(lambda c: c.x if c is not None else np.nan)
    y = centroids.apply(lambda c: c.y if c is not None else np.nan)

    grid_x = np.floor(x / grid_size).astype("Int64")
    grid_y = np.floor(y / grid_size).astype("Int64")

    groups = grid_x.astype(str) + "_" + grid_y.astype(str)
    return groups, x, y

In [6]:
def add_neighbor_missing_indicators(df):
    df = df.copy()

    neighbor_stat_cols = [
        c for c in df.columns
        if c.startswith("neighbor_height_")
    ]

    for col in neighbor_stat_cols:
        ind_col = f"has_{col}"
        df[ind_col] = df[col].notna().astype(int)
        df[col] = df[col].fillna(0)

    return df

In [7]:
def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "rmse": rmse,
        "mae": mae,
        "mse": mse,
        "r2": r2,
    }

In [31]:
def make_feature_sets():
    baseline = [
        "median_stairs_b",
        "median_avg_floor_height_b",
        "union_area_all",
        "mode_purpose_b",
    ]

    full = [
        "match_type",
        "match_confidence",
        "geometry_source",
        "target_height_reliability",
        "n_a",
        "n_b",
        "n_edges_ab",
        "sum_area_a",
        "sum_area_b",
        "union_area_a",
        "union_area_b",
        "union_area_all",
        "n_b_with_height",
        "median_height_b",
        "median_stairs_b",
        "median_avg_floor_height_b",
        "mode_purpose_b",

        "n_neighbors_50m",
        "n_neighbors_obs_height_50m",
        "neighbor_height_mean_50m",
        "neighbor_height_median_50m",
        "neighbor_height_min_50m",
        "neighbor_height_max_50m",
        "neighbor_height_std_50m",
        "neighbor_height_q25_50m",
        "neighbor_height_q75_50m",

        "n_neighbors_100m",
        "n_neighbors_obs_height_100m",
        "neighbor_height_mean_100m",
        "neighbor_height_median_100m",
        "neighbor_height_min_100m",
        "neighbor_height_max_100m",
        "neighbor_height_std_100m",
        "neighbor_height_q25_100m",
        "neighbor_height_q75_100m",

        "has_neighbor_height_mean_50m",
        "has_neighbor_height_median_50m",
        "has_neighbor_height_min_50m",
        "has_neighbor_height_max_50m",
        "has_neighbor_height_std_50m",
        "has_neighbor_height_q25_50m",
        "has_neighbor_height_q75_50m",

        "has_neighbor_height_mean_100m",
        "has_neighbor_height_median_100m",
        "has_neighbor_height_min_100m",
        "has_neighbor_height_max_100m",
        "has_neighbor_height_std_100m",
        "has_neighbor_height_q25_100m",
        "has_neighbor_height_q75_100m",
    ]

    return {
        "baseline": baseline,
        "full": full,
    }

## Этап 3 - Построение пайплайна модели

**Загрузка и предварительная фильтрация данных**

После загрузки parquet-файла из него оставляются только заранее определенные рабочие столбцы. Затем выборка ограничивается объектами, для которых высота действительно наблюдалась, то есть известна как фактическое значение, пригодное для обучения модели.

Дополнительно удаляются признаки, отражающие исключительно технические характеристики геометрического сопоставления между источниками. Они могут быть полезны для анализа самого процесса мерджа, но не всегда являются содержательно интерпретируемыми предикторами высоты здания.

---

**Добавление индикаторов пропусков и пространственной разметки объектов**

После фильтрации к данным добавляются бинарные признаки, отражающие наличие или отсутствие соседских статистик. Затем для каждого объекта вычисляется пространственная группа на основе его геометрии.

Также сохраняются координаты центроида, что удобно для последующей диагностики качества группировки и визуального контроля пространственного разбиения. Объекты без корректно определенной группы исключаются, так как без них невозможно выполнить групповую валидацию.

---

**Определение категориальных признаков для CatBoost**

Одним из преимуществ CatBoost является встроенная поддержка категориальных переменных. Поэтому на этом шаге определяется, какие столбцы в текущем наборе признаков имеют категориальный тип, и вычисляются их позиции в таблице признаков.

Эта информация затем передается модели напрямую. За счет этого не требуется вручную выполнять one-hot-кодирование или ordinal-кодирование категориальных признаков, как это обычно делается в классических пайплайнах на основе `scikit-learn`.

---

**Построение модели CatBoostRegressor**

На этом этапе создается функция для инициализации модели `CatBoostRegressor` с заданными гиперпараметрами. В модели фиксируется зерно случайности для воспроизводимости, а в качестве основной функции потерь и метрики оптимизации используется `RMSE`.

Выбор CatBoost здесь оправдан тем, что алгоритм хорошо работает на неоднородных табличных данных, умеет естественно учитывать категориальные признаки и часто показывает высокое качество без сложной предварительной кодировки признаков.

---

**Реализация групповой кросс-валидации для CatBoost**

Для оценки модели используется `GroupKFold`, где разбиение выполняется по пространственным группам, а не по отдельным объектам. На каждом фолде модель обучается на одной части пространственных кластеров и тестируется на другой.

Перед обучением категориальные признаки приводятся к строковому виду, а пропущенные значения заполняются специальным маркером. Это делается для того, чтобы CatBoost корректно интерпретировал категориальные столбцы и не сталкивался с неоднозначностями типов.

Для каждого фолда сохраняются метрики качества, а также формируются out-of-fold предсказания, которые могут использоваться далее для анализа ошибок или построения ансамблей.

---

**Задание набора гиперпараметров CatBoost**

На этом шаге формируется список тестируемых конфигураций модели. Для CatBoost рассматриваются параметры числа итераций, скорости обучения, глубины деревьев, коэффициента регуляризации листьев и минимального числа объектов в листе.

В отличие от полного перебора большой сетки, здесь используется компактный набор заранее выбранных конфигураций. Такой подход позволяет быстро сравнить несколько разумных вариантов модели, не увеличивая чрезмерно время эксперимента.

---

**Перебор наборов признаков и параметров модели**

Далее выполняется основной цикл экспериментов. Для каждого набора признаков формируется матрица признаков, после чего модель CatBoost обучается и оценивается на всех заданных наборах гиперпараметров.

По каждой комбинации сохраняются средние метрики по всем фолдам, а также out-of-fold предсказания. Это позволяет не только сравнить итоговые значения качества, но и при необходимости вернуться к конкретному запуску и изучить его поведение более подробно.

---

**Формирование итоговой таблицы результатов**

После завершения всех запусков результаты собираются в единую таблицу и сортируются по метрикам `RMSE` и `MAE`. Такая сортировка позволяет сразу увидеть лучшие конфигурации модели и понять, какой набор признаков оказался наиболее информативным.

Итоговая таблица служит основой для сравнительного анализа: по ней можно оценить, насколько использование полного набора признаков превосходит базовый вариант и насколько чувствительна модель к изменению гиперпараметров.


In [9]:
df = pd.read_parquet(DATA_PATH)
df = df[main_cols].copy()

df = df[df["target_height_is_observed"] == 1].copy()

drop_match_metric_cols = [
    "max_iou",
    "mean_iou",
    "max_overlap_a",
    "max_overlap_b",
    "min_dist_m",
]
df = df.drop(columns=drop_match_metric_cols)

df = add_neighbor_missing_indicators(df)

df["spatial_group"], df["centroid_x"], df["centroid_y"] = build_spatial_groups(
    df,
    geom_col="rep_geometry",
    grid_size=GRID_SIZE,
)

df = df[df["spatial_group"].notna()].copy()

print(df.shape)
df.head()

(101563, 60)


,component_id,match_type,match_confidence,geometry_source,target_height,target_height_is_observed,target_height_source,target_height_source_detail,target_height_reliability,n_a,...,has_neighbor_height_mean_100m,has_neighbor_height_median_100m,has_neighbor_height_min_100m,has_neighbor_height_max_100m,has_neighbor_height_std_100m,has_neighbor_height_q25_100m,has_neighbor_height_q75_100m,spatial_group,centroid_x,centroid_y
0,1,n:1,high,A,4.50,1,B_observed,B_from_nto1_match,medium,8,...,1,1,1,1,1,1,1,673_6635,673859.539664,6.635505e+06
1,2,n:1,high,A,4.50,1,B_observed,B_from_nto1_match,medium,5,...,1,1,1,1,1,1,1,673_6635,673885.218595,6.635511e+06
2,3,n:n,high,A,60.00,1,B_observed,B_from_nton_match,medium,12,...,1,1,1,1,1,1,1,677_6640,677022.530648,6.640407e+06
5,6,n:n,high,B,48.00,1,B_observed,B_from_nton_match,medium,18,...,1,1,1,1,1,1,1,678_6638,678532.626405,6.638733e+06
6,7,n:n,high,B,68.25,1,B_observed,B_from_nton_match,medium,3,...,1,1,1,1,1,1,1,678_6640,678893.429480,6.640099e+06


In [ ]:
def get_catboost_feature_info(X):
    categorical_cols = X.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    cat_features = [X.columns.get_loc(col) for col in categorical_cols]
    return categorical_cols, cat_features

In [11]:
def build_catboost_model(cb_params):
    model = CatBoostRegressor(
        random_seed=RANDOM_STATE,
        loss_function="RMSE",
        eval_metric="RMSE",
        verbose=0,
        **cb_params,
    )
    return model

In [22]:
def cross_validate_catboost_with_groups(
    X,
    y,
    groups,
    cb_params,
    n_splits=5,
    return_oof=True,
):
    cv = GroupKFold(n_splits=n_splits)

    fold_rows = []
    oof_pred = np.full(len(X), np.nan, dtype=float)

    _, cat_features = get_catboost_feature_info(X)

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        for col in X_train.select_dtypes(include=["object", "category", "string"]).columns:
            X_train[col] = X_train[col].astype(str).fillna("missing")
            X_test[col] = X_test[col].astype(str).fillna("missing")

        model = build_catboost_model(cb_params)
        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
        )

        pred = model.predict(X_test)

        if return_oof:
            oof_pred[test_idx] = pred

        metrics = compute_metrics(y_test, pred)
        metrics["fold"] = fold
        fold_rows.append(metrics)

    folds_df = pd.DataFrame(fold_rows)
    mean_metrics = folds_df[["rmse", "mae", "mse", "r2"]].mean().to_dict()

    return folds_df, mean_metrics, oof_pred

In [33]:
cb_param_list = [
    {
        "iterations": 300,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 3,
        "min_data_in_leaf": 10,
    },
    {
        "iterations": 300,
        "learning_rate": 0.05,
        "depth": 8,
        "l2_leaf_reg": 5,
        "min_data_in_leaf": 30,
    },
]

print(f"Всего комбинаций CatBoost: {len(cb_param_list)}")
cb_param_list[:3]

Всего комбинаций CatBoost: 2


[{'iterations': 300,
  'learning_rate': 0.05,
  'depth': 6,
  'l2_leaf_reg': 3,
  'min_data_in_leaf': 10},
 {'iterations': 300,
  'learning_rate': 0.05,
  'depth': 8,
  'l2_leaf_reg': 5,
  'min_data_in_leaf': 30}]

In [32]:
feature_sets = make_feature_sets()

target_col = "target_height"
group_col = "spatial_group"

print("Наборы признаков:", list(feature_sets.keys()))
print("target_col =", target_col)
print("group_col =", group_col)

Наборы признаков: ['baseline', 'full']
target_col = target_height
group_col = spatial_group


In [34]:
cb_results = []
cb_oof_store = {}

y = df[target_col].reset_index(drop=True)
groups = df[group_col].reset_index(drop=True)

for set_name, feature_cols in feature_sets.items():
    print(f"\n===== Feature set: {set_name} =====")

    X = df[feature_cols].reset_index(drop=True)

    for params in cb_param_list:
        print(f"CatBoost params: {params}")

        folds_df, mean_metrics, oof_pred = cross_validate_catboost_with_groups(
            X=X,
            y=y,
            groups=groups,
            cb_params=params,
            n_splits=N_SPLITS,
            return_oof=True,
        )

        row = {
            "model_type": "catboost",
            "feature_set": set_name,
            **params,
            **mean_metrics,
        }
        cb_results.append(row)

        run_key = (
            "catboost",
            set_name,
            tuple(sorted(params.items()))
        )
        cb_oof_store[run_key] = oof_pred

cb_results_df = (
    pd.DataFrame(cb_results)
    .sort_values(["rmse", "mae"], ascending=[True, True])
    .reset_index(drop=True)
)

cb_results_df.head(20)


===== Feature set: baseline =====
CatBoost params: {'iterations': 300, 'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 3, 'min_data_in_leaf': 10}
CatBoost params: {'iterations': 300, 'learning_rate': 0.05, 'depth': 8, 'l2_leaf_reg': 5, 'min_data_in_leaf': 30}

===== Feature set: full =====
CatBoost params: {'iterations': 300, 'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 3, 'min_data_in_leaf': 10}
CatBoost params: {'iterations': 300, 'learning_rate': 0.05, 'depth': 8, 'l2_leaf_reg': 5, 'min_data_in_leaf': 30}


,model_type,feature_set,iterations,learning_rate,depth,l2_leaf_reg,min_data_in_leaf,rmse,mae,mse,r2
0,catboost,full,300,0.05,6,3,10,0.686816,0.068745,1.470125,0.989195
1,catboost,full,300,0.05,8,5,30,0.744293,0.107251,1.441920,0.989331
2,catboost,baseline,300,0.05,6,3,10,2.606065,0.370035,7.358498,0.935106
3,catboost,baseline,300,0.05,8,5,30,2.610122,0.376256,7.371213,0.935018


In [35]:
best_cb_row = cb_results_df.iloc[0].to_dict()
best_cb_row

{'model_type': 'catboost',
 'feature_set': 'full',
 'iterations': 300,
 'learning_rate': 0.05,
 'depth': 6,
 'l2_leaf_reg': 3,
 'min_data_in_leaf': 10,
 'rmse': 0.6868158520865602,
 'mae': 0.06874540024031224,
 'mse': 1.4701254119029943,
 'r2': 0.9891952577943129}

## Объяснение результатов

По итогам эксперимента видно, что модель CatBoost значительно лучше работает на полном наборе признаков, чем на базовом. Это означает, что для предсказания высоты здания существенно важны не только общие характеристики самого объекта, но и более широкий контекст: признаки сопоставления, агрегаты по геометрии и особенно статистики по соседним зданиям.

Разрыв между `baseline` и `full` по `RMSE` и `MAE` достаточно велик, чтобы считать его содержательно значимым. Следовательно, пространственное окружение и дополнительные структурные признаки действительно вносят сильный вклад в качество прогноза.

При этом различие между двумя протестированными конфигурациями CatBoost внутри одного и того же набора признаков заметно меньше, чем различие между самими наборами признаков. Это указывает на важный вывод: в данной задаче качество сильнее определяется информативностью признакового пространства, чем тонкой подстройкой гиперпараметров модели.